In [1]:
import os
import numpy as np
import pandas as pd
import torch
from typing import Dict

_possible_base_dirs = [
    "data/hms-harmful-brain-activity-classification",
    "/kaggle/input/hms-harmful-brain-activity-classification",
    "./data/hms-harmful-brain-activity-classification",
]
BASE_PATH = next((d for d in _possible_base_dirs if os.path.isdir(d)), None)
if BASE_PATH is None:
    raise FileNotFoundError("Could not locate the competition data directory.")

TRAIN_PATH = os.path.join(BASE_PATH, "train.csv")
TEST_PATH = os.path.join(BASE_PATH, "test.csv")
SUBMISSION_PATH = "submission.csv"


def safe_read_csv(path: str) -> pd.DataFrame:
    """Attempt several common prefixes to locate a CSV file."""
    possible = [
        path,
        os.path.abspath(path),
        os.path.join(".", path),
        os.path.join("input", os.path.basename(path)),
        os.path.join(".", os.path.basename(path)),
    ]
    for p in possible:
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(f"Could not locate {path} in any of {possible}")


df_train = safe_read_csv(TRAIN_PATH)
df_test = safe_read_csv(TEST_PATH)

print(f"Train rows: {len(df_train)}, Test rows: {len(df_test)}")
print()




Train rows: 96950, Test rows: 9850



In [2]:
LABELS = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]

# Reduce Laplace smoothing to zero so empirical vote ratios are used directly
ALPHA = 0.0

global_votes = df_train[LABELS].astype(np.float64).sum(axis=0) + ALPHA
total_votes = global_votes.sum()
PRIOR_PROBS = (global_votes / total_votes).astype(np.float32)

UNIFORM_PROB = np.full(len(LABELS), 1.0 / len(LABELS), dtype=np.float32)

eeg_group = df_train.groupby("eeg_id")[LABELS].sum().reset_index()
EIG_PRIOR_DICT: Dict[int, np.ndarray] = {}
EIG_VOTE_SUM_DICT: Dict[int, float] = {}
for _, row in eeg_group.iterrows():
    eid = int(row["eeg_id"])
    votes = row[LABELS].astype(np.float64).to_numpy() + ALPHA
    s = votes.sum()
    # Guard against division by zero; fallback to global prior
    probs = (votes / s).astype(np.float32) if s > 0 else PRIOR_PROBS.copy()
    EIG_PRIOR_DICT[eid] = probs
    EIG_VOTE_SUM_DICT[eid] = float(s)

patient_group = df_train.groupby("patient_id")[LABELS].sum().reset_index()
PAT_PRIOR_DICT: Dict[int, np.ndarray] = {}
PAT_VOTE_SUM_DICT: Dict[int, float] = {}
for _, row in patient_group.iterrows():
    pid = int(row["patient_id"])
    votes = row[LABELS].astype(np.float64).to_numpy() + ALPHA
    s = votes.sum()
    probs = (votes / s).astype(np.float32) if s > 0 else PRIOR_PROBS.copy()
    PAT_PRIOR_DICT[pid] = probs
    PAT_VOTE_SUM_DICT[pid] = float(s)

# Remove uniform blending to let empirical priors dominate
UNIFORM_BLEND = 0.0




In [3]:
@torch.no_grad()
def gen_ensemble_pred(eeg_id: int, patient_id: int) -> np.ndarray:
    """
    Weighted combination of per‑eeg and per‑patient empirical distributions,
    using their respective vote counts as weights. Falls back to whichever
    prior is available, then to the global prior. No uniform blend is applied.
    The resulting vector is renormalised to sum to 1.
    """
    eig_exists = eeg_id in EIG_PRIOR_DICT
    pat_exists = patient_id in PAT_PRIOR_DICT

    if eig_exists and pat_exists:
        s_eeg = EIG_VOTE_SUM_DICT[eeg_id]
        s_pat = PAT_VOTE_SUM_DICT[patient_id]
        prob = (s_eeg * EIG_PRIOR_DICT[eeg_id] + s_pat * PAT_PRIOR_DICT[patient_id]) / (
            s_eeg + s_pat
        )
    elif eig_exists:
        prob = EIG_PRIOR_DICT[eeg_id].copy()
    elif pat_exists:
        prob = PAT_PRIOR_DICT[patient_id].copy()
    else:
        prob = PRIOR_PROBS.copy()

    # No uniform blend; only renormalise to guard against numerical issues
    prob = prob / prob.sum()
    return prob




In [4]:
preds = []
for _, row in df_test.iterrows():
    eid = int(row["eeg_id"])
    pid = int(row["patient_id"])
    prob = gen_ensemble_pred(eid, pid)
    preds.append(prob)

pred_array = np.vstack(preds)  # (n_test, n_classes)

submission_df = pd.DataFrame(pred_array, columns=LABELS)
submission_df.insert(0, "eeg_id", df_test["eeg_id"])
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Submission written to {SUBMISSION_PATH}")
print("First rows:")
print(submission_df.head())

Submission written to submission.csv
First rows:
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  2578018731      0.001616  0.016162  0.564848   0.006061   0.185051   
1  2578018731      0.001616  0.016162  0.564848   0.006061   0.185051   
2  2578018731      0.001616  0.016162  0.564848   0.006061   0.185051   
3  2578018731      0.001616  0.016162  0.564848   0.006061   0.185051   
4  1706196508      0.114686  0.300330  0.005776   0.138614   0.019802   

   other_vote  
0    0.226263  
1    0.226263  
2    0.226263  
3    0.226263  
4    0.420792  
